# Files Utilities (`files_utilities`)

El módulo `files_utilities` (`ds_utils.io.files_utilities`) proporciona funciones para importar y exportar datos en diferentes formatos de archivo.

* **`import_csv_to_df`**: importa archivos CSV a DataFrames de Pandas, permite configurar los parámetros de lectura y mostrar opcionalmente una previsualización del DataFrame.
* **`export_df_to_csv`**: exporta DataFrames a archivos CSV, crea automáticamente los directorios de destino si no existen y permite configurar las opciones de exportación y controlar la sobrescritura de archivos existentes.
* **`import_parquet_to_df`**: importa archivos Parquet a DataFrames de Pandas mediante PyArrow y permite mostrar opcionalmente una previsualización del DataFrame.
* **`export_df_to_parquet`**: exporta DataFrames a archivos Parquet mediante Pandas y PyArrow, crea automáticamente los directorios de destino si no existen y permite controlar la sobrescritura de archivos existentes.
* **`import_json_to_python`**: importa archivos JSON y devuelve el objeto Python correspondiente a su contenido.
* **`export_python_to_json`**: serializa objetos Python compatibles con JSON y los guarda en archivos, con opciones para configurar la serialización.

Este notebook muestra ejemplos de importación y exportación de datos en los formatos CSV, Parquet y JSON, y elimina los archivos generados al finalizar.

In [31]:
# AUTOMATIC RELOAD OF EXTERNAL MODULES
%load_ext autoreload
%autoreload 2

from pathlib import Path

from tempfile import TemporaryDirectory

import pandas as pd

from ds_utils.io.files_utilities import (
    export_df_to_csv,
    export_df_to_parquet,
    export_python_to_json,
    import_csv_to_df,
    import_parquet_to_df,
    import_json_to_python,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Create a sample DataFrame

Creamos un DataFrame de ejemplo con datos de sensores para comprobar que los datos se conservan después de exportarlos e importarlos.

In [32]:
df_original = pd.DataFrame({
    "timestamp": pd.date_range("2025-01-01", periods=5, freq="h"),
    "temperature": [21.5, 22.1, 20.8, 21.9, 22.4],
    "sensor": ["A", "B", "A", "B", "A"],
})

display(df_original)

,timestamp,temperature,sensor
0,2025-01-01 00:00:00,21.5,A
1,2025-01-01 01:00:00,22.1,B
2,2025-01-01 02:00:00,20.8,A
3,2025-01-01 03:00:00,21.9,B
4,2025-01-01 04:00:00,22.4,A


## 2. Create a Temporary Directory

Crea un nuevo directorio temporal para almacenar los archivos generados durante los siguientes ejemplos.
La ruta se almacena en `temp_path`, mientras que `csv_path`, `parquet_path` y `json_path` definen las ubicaciones de los archivos correspondientes.
El directorio temporal se eliminará explícitamente al finalizar mediante `temp_path.cleanup()`.

In [33]:
temp_dir = TemporaryDirectory()
temp_path = Path(temp_dir.name)

csv_path = temp_path / "sensors.csv"
parquet_path = temp_path / "sensors.parquet"
json_path = temp_path / "data.json"

print(f"Temporary directory: {temp_path}")

print(f"CSV temporary directory: {csv_path}")
print(f"Parquet temporary directory: {parquet_path}")
print(f"JSON temporary directory: {json_path}")

Temporary directory: /tmp/tmpronugp_q
CSV temporary directory: /tmp/tmpronugp_q/sensors.csv
Parquet temporary directory: /tmp/tmpronugp_q/sensors.parquet
JSON temporary directory: /tmp/tmpronugp_q/data.json


## 3. Export a DataFrame to CSV

Utilizamos `export_csv` para guardar el DataFrame. El directorio de destino es temporal y se eliminara al finalizar.

Se utiliza `index=False` para evitar guardar el índice del DataFrame como una columna adicional.

In [34]:
export_df_to_csv(
    df_original,
    csv_path,
    index=False,
)

print(f"CSV exported successfully to: {csv_path.resolve()}")

CSV exported successfully to: /tmp/tmpronugp_q/sensors.csv


## 4. Import a DataFrame from CSV

Leemos el archivo generado mediante `import_csv` y mostramos su contenido.
Usamos `parse_dates` para convertir la columna 'timestamp' de tipo `object` a `datetime64[ns]`

In [35]:
df_imported = import_csv_to_df(
    csv_path,
    parse_dates=["timestamp"], # cambiar timestamp de tipo `object` a `datetime64[ns]`
)

display(df_imported)

,timestamp,temperature,sensor
0,2025-01-01 00:00:00,21.5,A
1,2025-01-01 01:00:00,22.1,B
2,2025-01-01 02:00:00,20.8,A
3,2025-01-01 03:00:00,21.9,B
4,2025-01-01 04:00:00,22.4,A


## 5. Validate the result

Comprobamos que el DataFrame importado coincide con el original.

In [36]:
pd.testing.assert_frame_equal(df_original, df_imported)

print("CSV export and import validated successfully.")

CSV export and import validated successfully.


## 6. Export a DataFrame to Parquet

Utilizamos `export_df_to_parquet` para guardar el DataFrame de ejemplo en formato Parquet utilizando el motor `pyarrow`.

In [37]:
export_df_to_parquet(
    df_original,
    parquet_path,
)

print(f"Parquet exported successfully to: {parquet_path.resolve()}")

Parquet exported successfully to: /tmp/tmpronugp_q/sensors.parquet


## 7. Import a DataFrame from Parquet

Leemos el archivo Parquet generado mediante `import_parquet_to_df` y mostramos su contenido.
El formato Parquet conserva los tipos de datos nativos del DataFrame original (como `datetime64[ns]`) sin requerir parseo manual de fechas.

In [38]:
df_parquet_imported = import_parquet_to_df(
    parquet_path,
)

display(df_parquet_imported)

,timestamp,temperature,sensor
0,2025-01-01 00:00:00,21.5,A
1,2025-01-01 01:00:00,22.1,B
2,2025-01-01 02:00:00,20.8,A
3,2025-01-01 03:00:00,21.9,B
4,2025-01-01 04:00:00,22.4,A


## 8. Validate the Parquet result

Comprobamos que el DataFrame importado desde Parquet coincide con el original.

In [39]:
pd.testing.assert_frame_equal(df_original, df_parquet_imported)

print("Parquet export and import validated successfully.")

Parquet export and import validated successfully.


## 9. Create a sample Python object for JSON

Definimos una estructura de datos nativa de Python (diccionario con metadatos y lista de sensores) para comprobar la exportación e importación en formato JSON.

In [40]:
data_original = {
    "project": "ds-utils",
    "version": "0.1.0",
    "sensors": [
        {"id": "A", "type": "temperature", "active": True},
        {"id": "B", "type": "temperature", "active": False},
    ],
    "metadata": {
        "location": "Warehouse 1",
        "sample_count": 5,
    },
}

display(data_original)

{'project': 'ds-utils',
 'version': '0.1.0',
 'sensors': [{'id': 'A', 'type': 'temperature', 'active': True},
  {'id': 'B', 'type': 'temperature', 'active': False}],
 'metadata': {'location': 'Warehouse 1', 'sample_count': 5}}

## 10. Export a Python object to JSON

Utilizamos `export_python_to_json` para guardar la estructura de datos en un archivo JSON con formato legible (indentado).

In [41]:
export_python_to_json(
    data_original,
    json_path,
)

print(f"JSON exported successfully to: {json_path.resolve()}")

JSON exported successfully to: /tmp/tmpronugp_q/data.json


## 11. Import a Python object from JSON

Leemos el archivo generado mediante `import_json_to_python` para recuperar el objeto Python deserializado.

In [42]:
data_imported = import_json_to_python(
    json_path,
)

display(data_imported)

{'project': 'ds-utils',
 'version': '0.1.0',
 'sensors': [{'id': 'A', 'type': 'temperature', 'active': True},
  {'id': 'B', 'type': 'temperature', 'active': False}],
 'metadata': {'location': 'Warehouse 1', 'sample_count': 5}}

## 12. Validate the JSON result

Comprobamos que el objeto Python deserializado es idéntico a la estructura de datos original.

In [43]:
assert data_original == data_imported

print("JSON export and import validated successfully.")

JSON export and import validated successfully.


## 13. Clean up generated files

Eliminamos la carpeta temporal creada para los archivos CSV, Parquet y JSON y verificamos su eliminación.

In [44]:
temp_dir.cleanup()

# Verify that the temporary files and directory have been removed.
print(f"CSV file successfully deleted: {not csv_path.exists()}")
print(f"Parquet file successfully deleted: {not parquet_path.exists()}")
print(f"JSON file successfully deleted: {not json_path.exists()}")
print(f"Temporary directory successfully deleted: {not temp_path.exists()}")

CSV file successfully deleted: True
Parquet file successfully deleted: True
JSON file successfully deleted: True
Temporary directory successfully deleted: True
